# KNative Messaging Library Demo

This notebook demonstrates the `messaging` library — a transport-agnostic
Python library for publishing and handling CloudEvents via a simple `MessageBus` API.

The bus runs inside the cluster alongside the KNative Broker, so events
published here flow through the real infrastructure:

```
Notebook → MessageBus → KNative Broker → Triggers → Subscribers
```

## 1. Setup — Create a MessageBus with KNative Transport

In [ ]:
import os
from messaging import MessageBus, CloudEvent, Disposition, MessageContext
from messaging.knative import KNativeEventingPublisher

BROKER_URL = os.environ.get(
    "BROKER_URL",
    "http://kafka-broker-ingress.knative-eventing.svc.cluster.local/default/default"
)

bus = MessageBus()
bus.configure(KNativeEventingPublisher(broker_url=BROKER_URL))

print(f"✅ MessageBus configured with KNative transport")
print(f"   Broker URL: {BROKER_URL}")

## 2. Publish a CloudEvent

Events are published as CloudEvents (CE spec v1.0). The library handles
the HTTP transport and CE envelope automatically.

The event will appear in the **Interactive Demo** tab (left panel) since
the Trigger routes `com.example.demo` events to the demo backend.

In [ ]:
event = CloudEvent(
    type="com.example.demo",
    source="/demo/jupyter",
    data={"message": "Hello from Jupyter!", "origin": "notebook"}
)

await bus.publish(event)
print(f"📤 Published event: {event.id}")
print(f"   Type: {event.type}")
print(f"   Source: {event.source}")
print(f"   → Check the Interactive Demo tab!")

## 3. Publish to Azure Service Bus (via Broker)

To route an event to ASB, publish with type `asb.outbound`.
The `broker-to-asb` Camel-K integration picks it up and forwards
it to the `knative-outbound` queue.

Watch the **ASB Explorer** panel on the right!

In [ ]:
asb_event = CloudEvent(
    type="asb.outbound",
    source="/demo/jupyter",
    data={"message": "Hello ASB from Jupyter!", "destination": "knative-outbound"}
)

await bus.publish(asb_event)
print(f"📤 Published to ASB via Broker: {asb_event.id}")
print(f"   → Check ASB Explorer → knative-outbound queue")

## 4. Create an Event Stream Display

The `EventStream` widget renders CloudEvents inline as they arrive.
We'll connect it to a handler in the next step.

In [ ]:
from event_stream import EventStream

stream = EventStream()
stream.display()

## 5. Register a Handler

The handler receives CloudEvents and pushes them to the stream widget.
This is exactly how a real microservice handles events — the only
difference is that here we display them instead of processing them.

In production, the handler is invoked via HTTP (KNative Trigger → FastAPI router).
Here we use `bus.dispatch()` to invoke it locally.

In [ ]:
@bus.handler("com.example.demo")
async def handle_demo(event: CloudEvent, ctx: MessageContext) -> Disposition:
    """Handle incoming demo events and display them."""
    stream.append(event)
    return Disposition.COMPLETE

print("✅ Handler registered for: com.example.demo")
print("   Events will appear in the stream above ↑")

## 6. Trigger the Handler

Now let's dispatch an event locally. It goes through `bus.dispatch()`,
which invokes our handler, which appends to the stream.

You can also publish to the real Broker — the event will flow through
KNative infrastructure and arrive at any subscriber with a matching Trigger.

In [ ]:
# Local dispatch — handler is invoked immediately
test_event = CloudEvent(
    type="com.example.demo",
    source="/demo/jupyter",
    data={"message": "Dispatched locally!", "origin": "notebook"}
)

result = await bus.dispatch(test_event)
print(f"Handler returned: {result.value}")
print("Check the Event Stream above ↑")

In [ ]:
# Publish to the REAL Broker — flows through KNative infrastructure
real_event = CloudEvent(
    type="com.example.demo",
    source="/demo/jupyter",
    data={"message": "Published to Broker!", "origin": "notebook"}
)

await bus.publish(real_event)
print(f"📤 Published {real_event.id} to Broker")
print("   → Check the Interactive Demo tab to see it arrive!")

## 7. Summary

```python
bus = MessageBus()
bus.configure(KNativeEventingPublisher(broker_url="http://..."))

@bus.handler("event.type")
async def my_handler(event, ctx):
    # process the event
    return Disposition.COMPLETE

# Mount in FastAPI:
app.include_router(bus.router, prefix="/events")

# Publish:
await bus.publish(CloudEvent(type="...", source="/...", data={...}))
```

The library is **transport-agnostic** — same handler code works with
KNative, Kafka, or any CloudEvents-compatible broker.

In [ ]:
await bus.close()
print("🔒 Done")